# BIOMASS L1C Stack Notebook
 
This notebook stacks scences:
- BIO_S1_STA__1S_20260212T165444_20260212T165505_T_G01_M02_C01_T040_F176_01_DQCVP2
- BIO_S1_STA__1S_20260215T165446_20260215T165507_T_G01_M02_C02_T040_F176_01_DQCVQG
- BIO_S1_STA__1S_20260218T165448_20260218T165508_T_G01_M02_C03_T040_F176_01_DQCVRQ
- BIO_S1_STA__1S_20260221T165450_20260221T165510_T_G01_M02_C04_T040_F176_01_DQCVSL
- BIO_S1_STA__1S_20260224T165451_20260224T165512_T_G01_M02_C05_T040_F176_01_DQCVTT
- BIO_S1_STA__1S_20260227T165453_20260227T165514_T_G01_M02_C06_T040_F176_01_DQCVVL
- BIO_S1_STA__1S_20260302T165455_20260302T165515_T_G01_M02_C07_T040_F176_01_DQCVXK



In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import bioqlk #this is from the BIOPAL toolbox, from https://github.com/BioPAL
from scipy.ndimage import uniform_filter
from scipy.ndimage import uniform_filter

POLS = ("HH", "HV", "VH", "VV")
STA_ROOT = Path(r"C:\Users\Femke.Graveland\OneDrive - ESA\Documents\DATA\1C Sahara") # Update this path to where the STA scenes are located
scene_dirs = sorted(p for p in STA_ROOT.iterdir() if p.is_dir() and "F138" in p.name and "S1" in p.name) #this filters the directories to only include those that start with "BIO_S1_STA__1S_", so it only includes the STA scenes. Adjust this filter if your STA scenes have a different naming convention (eg. S2)

def crop_to_shape(arr, shape):
    row0 = max(0, (arr.shape[0] - shape[0]) // 2)
    col0 = max(0, (arr.shape[1] - shape[1]) // 2)
    return arr[row0:row0 + shape[0], col0:col0 + shape[1]]

def enl(intensity):
    intensity = np.asarray(intensity, dtype=np.float64)
    return float((np.mean(intensity) ** 2) / (np.var(intensity) + 1e-12))

def calculate_covariance(im1, im2, looksr, looksa):
    """Compute covariance matrix with multilooking."""
    corr = uniform_filter(np.real(im1*np.conj(im2)), [looksa,looksr]) + 1j* \
                uniform_filter(np.imag(im1*np.conj(im2)), [looksa,looksr])
    return corr

print(f"Found {len(scene_dirs)} scenes")




In [ ]:
def _crop(arr, shape):
    return arr[:shape[0], :shape[1]]

def _parabolic_offset(left, center, right):
    denom = left - 2.0 * center + right
    if np.abs(denom) < 1e-12:
        return 0.0
    return 0.5 * (left - right) / denom

def _estimate_subpixel_shift(ref_img, mov_img):
    """Return (dy, dx, quality) to align mov_img to ref_img."""
    ref = np.asarray(ref_img, dtype=np.float32)
    mov = np.asarray(mov_img, dtype=np.float32)

    ref = ref - np.nanmedian(ref)
    mov = mov - np.nanmedian(mov)

    window = np.outer(np.hanning(ref.shape[0]), np.hanning(ref.shape[1]))
    ref = ref * window
    mov = mov * window

    f_ref = np.fft.fft2(ref)
    f_mov = np.fft.fft2(mov)
    cross = f_ref * np.conj(f_mov)
    cross /= np.maximum(np.abs(cross), 1e-12)
    corr = np.fft.ifft2(cross)
    corr_abs = np.abs(corr)

    y, x = np.unravel_index(np.argmax(corr_abs), corr_abs.shape)
    peak = float(corr_abs[y, x])

    if y > ref.shape[0] // 2:
        y -= ref.shape[0]
    if x > ref.shape[1] // 2:
        x -= ref.shape[1]

    y0 = int((y + ref.shape[0]) % ref.shape[0])
    x0 = int((x + ref.shape[1]) % ref.shape[1])
    y_prev = corr_abs[(y0 - 1) % ref.shape[0], x0]
    y_next = corr_abs[(y0 + 1) % ref.shape[0], x0]
    x_prev = corr_abs[y0, (x0 - 1) % ref.shape[1]]
    x_next = corr_abs[y0, (x0 + 1) % ref.shape[1]]
    y_sub = _parabolic_offset(y_prev, corr_abs[y0, x0], y_next)
    x_sub = _parabolic_offset(x_prev, corr_abs[y0, x0], x_next)

    quality = peak / (np.median(corr_abs) + 1e-12)
    return float(y + y_sub), float(x + x_sub), float(quality)

def _shift_complex(arr, dy, dx):
    """Shift a 2D complex array by fractional pixels using the Fourier shift theorem."""
    arr = np.asarray(arr, dtype=np.complex64)
    ny, nx = arr.shape
    fy = np.fft.fftfreq(ny)[:, None]
    fx = np.fft.fftfreq(nx)[None, :]
    phase = np.exp(-2j * np.pi * (fy * dy + fx * dx))
    return np.fft.ifft2(np.fft.fft2(arr) * phase).astype(np.complex64)

scene_data = []
for scene_path in scene_dirs:
    hh, hv, vh, vv, _, _ = bioqlk.load_data(str(scene_path), return_metadata=True)
    scene_data.append({
        "HH": hh.astype(np.complex64),
        "HV": hv.astype(np.complex64),
        "VH": vh.astype(np.complex64),
        "VV": vv.astype(np.complex64),
    })

target_shape = (
    min(s["HH"].shape[0] for s in scene_data),
    min(s["HH"].shape[1] for s in scene_data),
)
scene_data = [{pol: _crop(s[pol], target_shape) for pol in POLS} for s in scene_data]

ref_idx = len(scene_data) // 2
ref_hh_amp = np.abs(scene_data[ref_idx]["HH"]).astype(np.float32)

accum = {pol: np.zeros(target_shape, dtype=np.complex64) for pol in POLS}
weight_sum = np.zeros(target_shape, dtype=np.float32)
scene_shifts = []
scene_weights = []

for i, scene in enumerate(scene_data):
    mov_hh_amp = np.abs(scene["HH"]).astype(np.float32)
    dy, dx, quality = _estimate_subpixel_shift(ref_hh_amp, mov_hh_amp)
    scene_shifts.append((dy, dx))
    scene_weights.append(quality)

    shifted_weight = np.full(target_shape, quality, dtype=np.float32)
    weight_sum += shifted_weight

    for pol in POLS:
        shifted = _shift_complex(scene[pol], dy, dx)
        accum[pol] += shifted * shifted_weight

stack_complex = {pol: accum[pol] / np.maximum(weight_sum, 1e-6) for pol in POLS}

stack_amp = {pol: np.abs(stack_complex[pol]).astype(np.float32) for pol in POLS}
stack_phase = {pol: np.angle(stack_complex[pol]).astype(np.float32) for pol in POLS}

border = int(min(32, max(8, np.ceil(max(np.max(np.abs(scene_shifts)), 0.0)) + 8)))
if target_shape[0] > 2 * border and target_shape[1] > 2 * border:
    stack_complex = {pol: arr[border:-border, border:-border] for pol, arr in stack_complex.items()}
    stack_amp = {pol: arr[border:-border, border:-border] for pol, arr in stack_amp.items()}
    stack_phase = {pol: arr[border:-border, border:-border] for pol, arr in stack_phase.items()}

print(f"Reference scene index: {ref_idx}")
print(f"Target stacked shape : {target_shape}")
print(f"Border crop          : {border} px")
for i, ((dy, dx), quality) in enumerate(zip(scene_shifts, scene_weights)):
    print(f"Scene {i:02d}: shift (dy, dx)=({dy:+.3f}, {dx:+.3f})  quality={quality:.3f}")

stack = {pol: np.abs(stack_complex[pol]) ** 2 for pol in POLS}
print(f"Stacked HH ENL: {enl(stack['HH']):.3f}")

In [ ]:
# Apply multilooking to stacked phase for smoother visualization
looksize = 7  # Adjust this for more/less smoothing
stack_phase_multilook = {}

for pol in POLS:
    # Wrap phase to [-pi, pi] and multilook
    phase_wrapped = np.angle(np.exp(1j * stack_phase[pol]))
    stack_phase_multilook[pol] = uniform_filter(phase_wrapped, size=looksize)

print(f'Applied multilooking with kernel size {looksize} to stacked phase')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 10))
raw_db = 10.0 * np.log10(np.clip(np.abs(scene_data[ref_idx]["HH"]) ** 2, 1e-10, None))
stk_db = 10.0 * np.log10(np.clip(stack["HH"], 1e-10, None))
vmin, vmax = np.percentile(stk_db[np.isfinite(stk_db)], [5, 95])
axes[0].imshow(raw_db, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
axes[0].set_title("Reference HH intensity (dB)")
# axes[0].axis("off")
axes[1].imshow(stk_db, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
axes[1].set_title("Stacked HH intensity (dB)")
# axes[1].axis("off")
plt.tight_layout()
plt.colorbar(axes[0].images[0], ax=axes, fraction=0.02, pad=0.01, label="Intensity (dB)")
plt.show()

In [ ]:
#ROI
range = 1000
azimuth = 16000
r_min, r_max = range -300, range + 300
az_min, az_max = azimuth - 1500, azimuth + 1500
vmin, vmax = np.percentile(stk_db[np.isfinite(stk_db)], [10, 90])
fig, axes = plt.subplots(1, 2, figsize=(15, 10))
axes[0].imshow(raw_db, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
axes[0].set_title("Reference HH intensity (dB)")
# axes[0].axis("off")
axes[0].set_xlim(r_min, r_max)
axes[0].set_ylim(az_max, az_min)
# axes[0].set.xlim(0, raw_db.shape[1])
axes[1].imshow(stk_db, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
axes[1].set_title("Stacked HH intensity (dB)")
axes[1].set_xlim(r_min, r_max)
axes[1].set_ylim(az_max, az_min)
# axes[1].axis("off")
plt.tight_layout()
plt.colorbar(axes[0].images[0], ax=axes, fraction=0.02, pad=0.01, label="Intensity (dB)")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 22))
stk_db_all = {pol: 10.0 * np.log10(np.clip(stack[pol], 1e-10, None)) for pol in POLS}
vmin, vmax = np.percentile(np.concatenate([stk_db_all[pol][np.isfinite(stk_db_all[pol])] for pol in POLS]), [5, 95])
vminHV, vmaxHV = -35,-10 #ajust these values if the HV/VH images look too dark
axes[0, 0].imshow(stk_db_all["HH"], cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
axes[0, 0].set_title("Stacked HH intensity (dB)")
axes[0, 0].axis("off")
plt.colorbar(axes[0, 0].images[0], ax=axes[0, 0], fraction=0.02, pad=0.01, label="Intensity (dB)")

axes[0, 1].imshow(stk_db_all["HV"], cmap="gray", vmin=vminHV, vmax=vmaxHV, aspect="auto")
axes[0, 1].set_title("Stacked HV intensity (dB)")
axes[0, 1].axis("off")
plt.colorbar(axes[0, 1].images[0], ax=axes[0, 1], fraction=0.02, pad=0.01, label="Intensity (dB)")

axes[1, 0].imshow(stk_db_all["VH"], cmap="gray", vmin=vminHV, vmax=vmaxHV, aspect="auto")
axes[1, 0].set_title("Stacked VH intensity (dB)")
axes[1, 0].axis("off")
plt.colorbar(axes[1, 0].images[0], ax=axes[1, 0], fraction=0.02, pad=0.01, label="Intensity (dB)")

axes[1, 1].imshow(stk_db_all["VV"], cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
axes[1, 1].set_title("Stacked VV intensity (dB)")
axes[1, 1].axis("off")
plt.colorbar(axes[1, 1].images[0], ax=axes[1, 1], fraction=0.02, pad=0.01, label="Intensity (dB)")
plt.tight_layout()
plt.show()

In [ ]:
# Phase difference of stacked phase: VV - HH (wrapped to [-pi, pi])
vv_hh_phase_diff = np.angle(np.exp(1j * (stack_phase["VV"] - stack_phase["HH"])))

fig, ax = plt.subplots(figsize=(10, 16))
im = ax.imshow(vv_hh_phase_diff, cmap="seismic", vmin=-np.pi, vmax=np.pi, aspect="auto")
ax.set_title("Stacked phase difference: VV - HH (wrapped)")
ax.set_xlabel("Range (pixels)")
ax.set_ylabel("Azimuth (pixels)")
fig.colorbar(im, ax=ax, label="Phase difference (rad)")
plt.tight_layout()
plt.show()

print(f"Mean: {np.nanmean(vv_hh_phase_diff):+.3f} rad")
print(f"Std : {np.nanstd(vv_hh_phase_diff):.3f} rad")

In [ ]:
# Compute raw (reference) VV-HH phase difference and compare to stacked coherent phase difference
import numpy as np
# raw phase diff from the reference scene
raw_phi = np.angle(scene_data[ref_idx]['VV'] * np.conj(scene_data[ref_idx]['HH']))
# stacked coherent phase diff (already computed as vv_hh_phase_diff earlier)
if 'vv_hh_phase_diff' not in locals():
    vv_hh_phase_diff = np.angle(np.exp(1j * (stack_phase['VV'] - stack_phase['HH'])))

def _center_crop(arr, shape):
    sr = max(0, (arr.shape[0] - shape[0]) // 2)
    sc = max(0, (arr.shape[1] - shape[1]) // 2)
    return arr[sr:sr + shape[0], sc:sc + shape[1]]

shape_common = (min(raw_phi.shape[0], vv_hh_phase_diff.shape[0]), min(raw_phi.shape[1], vv_hh_phase_diff.shape[1]))
raw_c = _center_crop(raw_phi, shape_common)
stack_c = _center_crop(vv_hh_phase_diff, shape_common)

phase_diff_delta = np.angle(np.exp(1j * (raw_c - stack_c)))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
im0 = axes[0].imshow(raw_c, cmap='seismic', vmin=-np.pi, vmax=np.pi, aspect='auto')
axes[0].set_title('Raw (ref) VV - HH (rad)')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='rad')

im1 = axes[1].imshow(stack_c, cmap='seismic', vmin=-np.pi, vmax=np.pi, aspect='auto')
axes[1].set_title('Stacked VV - HH (rad)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label='rad')

v = np.percentile(np.abs(phase_diff_delta[np.isfinite(phase_diff_delta)]), 99)
im2 = axes[2].imshow(phase_diff_delta, cmap='bwr', vmin=-v, vmax=v, aspect='auto')
axes[2].set_title('Raw - Stacked phase diff (rad)')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, label='rad')

plt.tight_layout()
plt.show()

print(f'Mean: {np.nanmean(phase_diff_delta):+.3f} rad')
print(f'Std : {np.nanstd(phase_diff_delta):.3f} rad')

In [ ]:

POLS = ("HH", "HV", "VH", "VV")
stack_coherent = {}
for pol in POLS:
    stack_coherent[pol] = np.mean([scene[pol] for scene in scene_data], axis=0).astype(np.complex64)

stack_coh_phase = {pol: np.angle(stack_coherent[pol]).astype(np.float32) for pol in POLS}

phi_hh = np.angle(stack_coherent["HH"])
phi_vv = np.angle(stack_coherent["VV"])
phi_diff = np.angle(np.exp(1j * (phi_hh - phi_vv)))
phase_coherence = float(np.abs(np.mean(np.exp(1j * phi_diff))))
phase_spread = float(np.sqrt(max(0.0, -2.0 * np.log(max(phase_coherence, 1e-12)))))


print(f"Mean HH-VV phase coherence: {phase_coherence:.3f}")
print(f"HH-VV circular phase spread: {phase_spread:.3f} rad")

fig, axes = plt.subplots(1, 3, figsize=(18, 10))
axes[0].imshow(stack_phase["HH"], cmap="seismic", vmin=-np.pi, vmax=np.pi, aspect="auto")
axes[0].set_title("Coherent HH phase")
axes[0].axis("off")
axes[1].imshow(stack_phase["VV"], cmap="seismic", vmin=-np.pi, vmax=np.pi, aspect="auto")
axes[1].set_title("Coherent VV phase")
axes[1].axis("off")
axes[2].imshow(stack_c, cmap="seismic", vmin=-np.pi, vmax=np.pi, aspect="auto")
axes[2].set_title("HH - VV phase difference")
axes[2].axis("off")
plt.tight_layout()
cbar = plt.colorbar(axes[2].images[0], ax=axes, fraction=0.02, pad=0.01, label="Phase (rad)")
cbar.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
cbar.set_ticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
plt.show()

#here you can see that there is a phase interference pattern in the HH-VV phase difference. I also have a script that stacks the phase if you're interested in that!

In [ ]:
# Plot VV amplitude (dB) with a box and crosshair at (range, azimuth)
# Show the selected cross-section location on a phase-difference image (HH - VV)
range = 1000
azimuth = 15500

r_min, r_max = range -300, range + 300
az_min, az_max = azimuth - 1500, azimuth + 1500
r_start = 650
r_end = 780
crosspoint = azimuth
amin, amax = azimuth - 3000, azimuth + 3000
rangemin, rangemax = range - 200, range + 200
vv_view_db = stk_db

# Contrast stretch
valid_vv = np.isfinite(vv_view_db) & (vv_view_db > -80)
if np.any(valid_vv):
    vmin_vv, vmax_vv = np.percentile(vv_view_db[valid_vv], [2, 98])
else:
    vmin_vv, vmax_vv = -60, 0

# Bounds checks
h, w = vv_view_db.shape
if not (0 <= range < w and 0 <= azimuth < h):
    raise IndexError(f"(range, azimuthh)=({range}) out of bounds for shape={vv_view_db.shape}")

fig, ax = plt.subplots(figsize=(10, 20))
im = ax.imshow(vv_view_db, cmap="gray", vmin=vmin_vv, vmax=0, aspect="auto")

# Crosshair and point
ax.axvline(range,color="hotpink", lw=1.5, label=f"range={range}")
ax.axhline(azimuth, color="purple", lw=1.5, label=f"azimuth={azimuth}")
ax.scatter(range, azimuth, c="yellow", edgecolors="black", zorder=3)

# Box defined by amin/amax and rangemin/rangemax
ax.plot(
    [rangemin, rangemax, rangemax, rangemin, rangemin],
    [amin, amin, amax, amax, amin],
    color="cyan",
    lw=2,
    label=f"box: x[{rangemin},{rangemax}], y[{amin},{amax}]"
)

ax.set_title(f"VV amplitude with ROI")
ax.set_xlabel("Range pixel")
ax.set_ylabel("Azimuth pixel")
ax.legend(loc="upper right")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="VV amplitude (dB)")
plt.tight_layout()
plt.show()

In [ ]:

# Cross-section at range index 20: phase difference (HH-VV) vs azimuth index
# range = 600  # range bin index
range = 1000
azimuth = 15500

r_min, r_max = range -300, range + 300
az_min, az_max = azimuth - 1500, azimuth + 1500
r_start = 650
r_end = 780
crosspoint = azimuth
amin, amax = azimuth - 3000, azimuth + 3000
rangemin, rangemax = range - 200, range + 200

range = 1030
azimuth = 15500
r_min, r_max = range -150, range + 150
az_min, az_max = azimuth - 750, azimuth + 750
r_start = 1020
r_end = 1040
az_start = 15300
az_end = 15600
crosspoint = azimuth

phase_cs = stack_c[azimuth, :]

valid_cs = np.isfinite(phase_cs)

az_idx = np.arange(stack_c.shape[1], dtype=int)
x_cs = az_idx[valid_cs]
y_cs = phase_cs[valid_cs]

plt.figure(figsize=(8, 5))
plt.plot(x_cs, y_cs, ".", ms=2, alpha=0.35, label="pixels")

# Smoothed trendline + min/max on the trend
if x_cs.size >= 5:
    win = min(501, x_cs.size if x_cs.size % 2 == 1 else x_cs.size - 1)  # odd
    win = max(5, win)
    if win % 2 == 0:
        win -= 1

    kernel = np.ones(win, dtype=np.float32) / win
    y_trend = np.convolve(y_cs.astype(np.float32), kernel, mode="same")

    plt.plot(x_cs, y_trend, "r-", lw=2, label=f"smoothed trend (win={win})")

    i_min = int(np.nanargmin(y_trend))
    i_max = int(np.nanargmax(y_trend))

    plt.scatter(x_cs[i_min], y_trend[i_min], c="cyan", s=35, zorder=3,
                label=f"trend min = {y_trend[i_min]:.2f} rad")
    plt.scatter(x_cs[i_max], y_trend[i_max], c="yellow", s=35, zorder=3,
                label=f"trend max = {y_trend[i_max]:.2f} rad")

    plt.axhline(y_trend[i_min], color="cyan", ls="--", lw=1, alpha=0.7)
    plt.axhline(y_trend[i_max], color="yellow", ls="--", lw=1, alpha=0.7)
    plt.xlim(rangemin, rangemax)
    plt.ylim(-0.1, 0.1)
    plt.axvline(range, color="purple", lw=1.5, label=f"crosspoint @ {range}")
    plt.axvline(r_start, color="darkviolet", lw=1.5, label="area of paleochannel")
    plt.axvline(r_end, color="darkviolet", lw=1.5)
    plt.axhspan(az_start, az_end, alpha=0.15, color="violet")

plt.title(f"Phase difference (HH-VV) vs range index at azimuth index {azimuth}")

#ROI


# phase_view = stack_c[az_min:az_max, r_min:r_max]

plt.figure(figsize=(6, 10))
plt.imshow(stack_c, cmap="seismic", vmin=-np.pi, vmax=np.pi, aspect="auto")
plt.axvline(range, color="hotpink", lw=1.5, label=f"cross-section @ range={range}")
plt.axvline(r_end, color="magenta", lw=1.5, label=f"area of paleochannel")
plt.axvline(r_start, color="magenta", lw=1.5)
plt.axhline(crosspoint, color="purple", lw=1.5, label=f"crosspoint @ {crosspoint}")
plt.axhline(az_start, color="darkviolet", lw=1.5, label="area of paleochannel")
plt.axhline(az_end, color="darkviolet", lw=1.5)
plt.axhspan(az_start, az_end, alpha=0.15, color="violet")
plt.axvspan(r_start, r_end, alpha=0.15, color="hotpink")
plt.title(f"Cross-section location in phase difference (HH-VV) ")
plt.xlabel("Range pixel")
plt.ylabel("Azimuth pixel")
plt.legend(loc="upper right")
cbar = plt.colorbar(fraction=0.03, pad=0.02, label="Phase difference (rad)")
cbar.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
cbar.set_ticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
plt.ylim(amax,amin)
plt.xlim(rangemin, rangemax)
plt.tight_layout()
plt.show()


plt.figure(figsize=(6, 10))
plt.imshow(stk_db, cmap="gray", vmin=-30, vmax=0, aspect="auto")
plt.axvline(range, color="hotpink", lw=1.5, label=f"cross-section @ range={range}")
plt.axvline(r_end, color="magenta", lw=1.5, label=f"area of paleochannel")
plt.axvline(r_start, color="magenta", lw=1.5)
plt.axhline(crosspoint, color="purple", lw=1.5, label=f"crosspoint @ {crosspoint}")
plt.axhline(az_start, color="darkviolet", lw=1.5, label="area of paleochannel")
plt.axhline(az_end, color="darkviolet", lw=1.5)
plt.axhspan(az_start, az_end, alpha=0.15, color="violet")
plt.axvspan(r_start, r_end, alpha=0.15, color="hotpink")
plt.title(f"Cross-section location in phase difference (HH-VV) ")
plt.xlabel("Range pixel")
plt.ylabel("Azimuth pixel")
plt.legend(loc="upper right")
# cbar = plt.colorbar(fraction=0.03, pad=0.02, label="Phase difference (rad)")
# cbar.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
# cbar.set_ticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
plt.ylim(amax,amin)
plt.xlim(rangemin, rangemax)
plt.tight_layout()
plt.show()



In [ ]:
# =============================================================================
# ABSOLUTE PHASE DIFFERENCE |HH - VV|
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------

range_px = 1030
azimuth_px = 15500

r_min, r_max = range_px - 150, range_px + 150
az_min, az_max = azimuth_px - 750, azimuth_px + 750

r_start = 1020
r_end = 1040

az_start = 15300
az_end = 15600

crosspoint = azimuth_px

amin, amax = azimuth_px - 3000, azimuth_px + 3000
rangemin, rangemax = range_px - 200, range_px + 200

# Optional white-to-red colormap
white_red = LinearSegmentedColormap.from_list(
    "white_red",
    ["white", "red"]
)

# -----------------------------------------------------------------------------
# ABSOLUTE PHASE DIFFERENCE IMAGE
# -----------------------------------------------------------------------------

abs_phase = np.abs(stack_c)

# -----------------------------------------------------------------------------
# CROSS-SECTION AT FIXED AZIMUTH
# -----------------------------------------------------------------------------

phase_cs = abs_phase[azimuth_px, :]

valid_cs = np.isfinite(phase_cs)

range_idx = np.arange(stack_c.shape[1], dtype=int)

x_cs = range_idx[valid_cs]
y_cs = phase_cs[valid_cs]

plt.figure(figsize=(8, 5))

plt.plot(
    x_cs,
    y_cs,
    ".",
    ms=2,
    alpha=0.35,
    label="pixels"
)

# -----------------------------------------------------------------------------
# SMOOTHED TREND
# -----------------------------------------------------------------------------

if x_cs.size >= 5:

    win = min(
        501,
        x_cs.size if x_cs.size % 2 == 1 else x_cs.size - 1
    )

    win = max(5, win)

    if win % 2 == 0:
        win -= 1

    kernel = np.ones(win, dtype=np.float32) / win

    y_trend = np.convolve(
        y_cs.astype(np.float32),
        kernel,
        mode="same"
    )

    plt.plot(
        x_cs,
        y_trend,
        "r-",
        lw=2,
        label=f"smoothed trend, win={win}"
    )

    i_min = int(np.nanargmin(y_trend))
    i_max = int(np.nanargmax(y_trend))

    plt.scatter(
        x_cs[i_min],
        y_trend[i_min],
        c="cyan",
        s=40,
        zorder=3,
        label=f"trend min = {y_trend[i_min]:.3f} rad"
    )

    plt.scatter(
        x_cs[i_max],
        y_trend[i_max],
        c="yellow",
        edgecolor="black",
        s=40,
        zorder=3,
        label=f"trend max = {y_trend[i_max]:.3f} rad"
    )

    plt.axhline(
        y_trend[i_min],
        color="cyan",
        ls="--",
        lw=1,
        alpha=0.7
    )

    plt.axhline(
        y_trend[i_max],
        color="yellow",
        ls="--",
        lw=1,
        alpha=0.7
    )

# -----------------------------------------------------------------------------
# CROSS-SECTION FORMATTING
# -----------------------------------------------------------------------------

plt.axvline(
    range_px,
    color="purple",
    lw=1.5,
    label=f"crosspoint @ range={range_px}"
)

plt.axvline(
    r_start,
    color="darkviolet",
    lw=1.5,
    label="area of paleochannel"
)

plt.axvline(
    r_end,
    color="darkviolet",
    lw=1.5
)

plt.axvspan(
    r_start,
    r_end,
    alpha=0.15,
    color="violet"
)

plt.xlim(rangemin, rangemax)
plt.ylim(0, np.pi)

plt.xlabel("Range pixel")
plt.ylabel("|Phase difference| (rad)")

plt.title(
    f"Absolute phase difference |HH-VV| vs range index at azimuth {azimuth_px}"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# =============================================================================
# ABSOLUTE PHASE DIFFERENCE MAP
# =============================================================================

plt.figure(figsize=(6, 10))

im = plt.imshow(
    abs_phase,
    cmap=white_red,      # or use cmap="Reds"
    vmin=0,
    vmax=np.pi,
    aspect="auto"
)

plt.axvline(
    range_px,
    color="hotpink",
    lw=1.5,
    label=f"cross-section @ range={range_px}"
)

plt.axvline(
    r_start,
    color="magenta",
    lw=1.5,
    label="area of paleochannel"
)

plt.axvline(
    r_end,
    color="magenta",
    lw=1.5
)

plt.axhline(
    crosspoint,
    color="purple",
    lw=1.5,
    label=f"crosspoint @ azimuth={crosspoint}"
)

plt.axhline(
    az_start,
    color="darkviolet",
    lw=1.5,
    label="area of paleochannel"
)

plt.axhline(
    az_end,
    color="darkviolet",
    lw=1.5
)

plt.axhspan(
    az_start,
    az_end,
    alpha=0.15,
    color="violet"
)

plt.axvspan(
    r_start,
    r_end,
    alpha=0.15,
    color="hotpink"
)

plt.title("Cross-section location in absolute phase difference |HH-VV|")
plt.xlabel("Range pixel")
plt.ylabel("Azimuth pixel")

cbar = plt.colorbar(
    im,
    fraction=0.03,
    pad=0.02,
    label="|Phase difference| (rad)"
)

cbar.set_ticks([0, np.pi / 2, np.pi])
cbar.set_ticklabels([
    "0",
    r"$\pi/2$",
    r"$\pi$"
])

plt.legend(loc="upper right")

plt.ylim(amax, amin)
plt.xlim(rangemin, rangemax)

plt.tight_layout()
plt.show()

# =============================================================================
# SAME LOCATION ON BACKSCATTER IMAGE
# =============================================================================

plt.figure(figsize=(6, 10))

plt.imshow(
    stk_db,
    cmap="gray",
    vmin=-30,
    vmax=0,
    aspect="auto"
)

plt.axvline(
    range_px,
    color="hotpink",
    lw=1.5,
    label=f"cross-section @ range={range_px}"
)

plt.axvline(
    r_start,
    color="magenta",
    lw=1.5,
    label="area of paleochannel"
)

plt.axvline(
    r_end,
    color="magenta",
    lw=1.5
)

plt.axhline(
    crosspoint,
    color="purple",
    lw=1.5,
    label=f"crosspoint @ azimuth={crosspoint}"
)

plt.axhline(
    az_start,
    color="darkviolet",
    lw=1.5,
    label="area of paleochannel"
)

plt.axhline(
    az_end,
    color="darkviolet",
    lw=1.5
)

plt.axhspan(
    az_start,
    az_end,
    alpha=0.15,
    color="violet"
)

plt.axvspan(
    r_start,
    r_end,
    alpha=0.15,
    color="hotpink"
)

plt.title("Cross-section location in backscatter image")
plt.xlabel("Range pixel")
plt.ylabel("Azimuth pixel")

plt.legend(loc="upper right")

plt.ylim(amax, amin)
plt.xlim(rangemin, rangemax)

plt.tight_layout()
plt.show()

In [ ]:


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 10))

# ==========================================================
# RANGE CROSS-SECTION (fixed azimuth)
# ==========================================================

phase_range = stack_c[azimuth, :]

valid = np.isfinite(phase_range)
x = np.arange(stack_c.shape[1])[valid]
y = phase_range[valid]

ax1.plot(x, y, ".", ms=2, alpha=0.35, color="black", label="pixels")

if x.size >= 5:

    win = min(501, x.size if x.size % 2 else x.size - 1)
    win = max(5, win)

    kernel = np.ones(win) / win
    y_trend = np.convolve(y, kernel, mode="same")

    ax1.plot(x, y_trend, "r-", lw=2, label=f"trend (win={win})")

    i_min = np.nanargmin(y_trend)
    i_max = np.nanargmax(y_trend)

    ax1.scatter(
        x[i_min], y_trend[i_min],
        c="cyan", s=40,
        label=f"min = {y_trend[i_min]:.2f} rad"
    )
    ax1.scatter(
        x[i_max], y_trend[i_max],
        c="yellow", s=40,
        edgecolor="k",
        label=f"max = {y_trend[i_max]:.2f} rad"
    )

ax1.axvline(
    range,
    color="hotpink",
    lw=2,
    label=f"crosspoint @ {range}"
)

ax1.axvline(
    r_start,
    color="magenta",
    lw=2,
    label="paleochannel"
)

ax1.axvline(r_end, color="magenta", lw=2)

ax1.axvspan(
    r_start,
    r_end,
    color="hotpink",
    alpha=0.15
)

ax1.set_xlim(rangemin, rangemax)
ax1.set_ylim(-0.1, 0.1)

ax1.set_title(
    f"HH-VV phase vs range\n(azimuth={azimuth})"
)

ax1.set_xlabel("Range index", color="purple")
ax1.set_ylabel("Phase difference (rad)")

ax1.spines["bottom"].set_color("purple")
ax1.tick_params(axis="x", colors="purple")

ax1.grid(alpha=0.3)

# ==========================================================
# AZIMUTH CROSS-SECTION (fixed range)
# ==========================================================

phase_az = stack_c[:, range]

valid = np.isfinite(phase_az)
x = np.arange(stack_c.shape[0])[valid]
y = phase_az[valid]

ax2.plot(x, y, ".", ms=2, alpha=0.35, color="black", label="pixels")

if x.size >= 5:

    win = min(501, x.size if x.size % 2 else x.size - 1)
    win = max(5, win)

    kernel = np.ones(win) / win
    y_trend = np.convolve(y, kernel, mode="same")

    ax2.plot(x, y_trend, "r-", lw=2, label=f"trend (win={win})")

    i_min = np.nanargmin(y_trend)
    i_max = np.nanargmax(y_trend)

    ax2.scatter(
        x[i_min], y_trend[i_min],
        c="cyan", s=40,
        label=f"min = {y_trend[i_min]:.2f} rad"
    )

    ax2.scatter(
        x[i_max], y_trend[i_max],
        c="yellow", s=40,
        edgecolor="k",
        label=f"max = {y_trend[i_max]:.2f} rad"
    )

ax2.axvline(
    azimuth,
    color="purple",
    lw=2,
    label=f"crosspoint @ {azimuth}"
)

ax2.axvline(
    az_start,
    color="darkviolet",
    lw=2,
    label="paleochannel"
)

ax2.axvline(
    az_end,
    color="darkviolet",
    lw=2
)

ax2.axvspan(
    az_start,
    az_end,
    color="purple",
    alpha=0.15
)

ax2.set_xlim(amin, amax)
ax2.set_ylim(-0.1, 0.1)

ax2.set_title(
    f"HH-VV phase vs azimuth\n(range={range})"
)

ax2.set_xlabel("Azimuth index", color="magenta")

ax2.spines["bottom"].set_color("magenta")
ax2.tick_params(axis="x", colors="magenta")

ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# Cross-section at range index 20: phase difference (HH-VV) vs azimuth index
# range = 600  # range bin index
range = 190
azimuth = 7980
rangemin, rangemax = 50,450
amin, amax = azimuth - 3000, azimuth + 3000
r_start = 120
r_end = 210
az_start = 7500
az_end = 8030
crosspoint = azimuth

phase_cs = stack_c[azimuth, :]

valid_cs = np.isfinite(phase_cs)

az_idx = np.arange(stack_c.shape[1], dtype=int)
x_cs = az_idx[valid_cs]
y_cs = phase_cs[valid_cs]

plt.figure(figsize=(8, 5))
plt.plot(x_cs, y_cs, ".", ms=2, alpha=0.35, label="pixels")

# Smoothed trendline + min/max on the trend
if x_cs.size >= 5:
    win = min(501, x_cs.size if x_cs.size % 2 == 1 else x_cs.size - 1)  # odd
    win = max(5, win)
    if win % 2 == 0:
        win -= 1

    kernel = np.ones(win, dtype=np.float32) / win
    y_trend = np.convolve(y_cs.astype(np.float32), kernel, mode="same")

    plt.plot(x_cs, y_trend, "r-", lw=2, label=f"smoothed trend (win={win})")

    i_min = int(np.nanargmin(y_trend))
    i_max = int(np.nanargmax(y_trend))

    plt.scatter(x_cs[i_min], y_trend[i_min], c="cyan", s=35, zorder=3,
                label=f"trend min = {y_trend[i_min]:.2f} rad")
    plt.scatter(x_cs[i_max], y_trend[i_max], c="yellow", s=35, zorder=3,
                label=f"trend max = {y_trend[i_max]:.2f} rad")

    plt.axhline(y_trend[i_min], color="cyan", ls="--", lw=1, alpha=0.7)
    plt.axhline(y_trend[i_max], color="yellow", ls="--", lw=1, alpha=0.7)
    plt.xlim(rangemin, rangemax)
    plt.ylim(-0.1, 0.1)
    plt.axvline(range, color="hotpink", lw=1.5, label=f"crosspoint @ {range}")
    plt.axvline(r_start, color="magenta", lw=1.5, label="area of paleochannel")
    plt.axvline(r_end, color="magenta", lw=1.5)
    plt.axvspan(r_start, r_end, alpha=0.15, color="hotpink")
    


plt.title(f"Phase difference (HH-VV) vs range index at azimuth index {azimuth}")
plt.xlabel("Range index")

ax = plt.gca()
ax.spines["bottom"].set_color("purple")
ax.spines["bottom"].set_linewidth(2)

# ax.tick_params(axis="x", colors="purple", width=2)

plt.legend()
plt.show()


#ROI


# phase_view = stack_c[az_min:az_max, r_min:r_max]

plt.figure(figsize=(6, 10))
plt.imshow(stack_c, cmap="seismic", vmin=-np.pi, vmax=np.pi, aspect="auto")
plt.axvline(range, color="hotpink", lw=1.5, label=f"cross-section @ range={range}")
plt.axvline(r_end, color="magenta", lw=1.5, label=f"area of paleochannel")
plt.axvline(r_start, color="magenta", lw=1.5)
plt.axhline(crosspoint, color="purple", lw=1.5, label=f"crosspoint @ {crosspoint}")
plt.axhline(az_start, color="darkviolet", lw=1.5, label="area of paleochannel")
plt.axhline(az_end, color="darkviolet", lw=1.5)
plt.axvspan(r_start, r_end, alpha=0.15, color="hotpink")
plt.title(f"Cross-section location in phase difference (HH-VV) ")
plt.xlabel("Range pixel")
plt.ylabel("Azimuth pixel")
plt.legend(loc="upper right")
cbar = plt.colorbar(fraction=0.03, pad=0.02, label="Phase difference (rad)")
cbar.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
cbar.set_ticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
plt.ylim(amax,amin)
plt.xlim(rangemin, rangemax)
plt.tight_layout()
plt.show()


plt.figure(figsize=(6, 10))
plt.imshow(stk_db, cmap="gray", vmin=-30, vmax=0, aspect="auto")
plt.axvline(range, color="hotpink", lw=1.5, label=f"cross-section @ range={range}")
plt.axvline(r_end, color="magenta", lw=1.5, label=f"area of paleochannel")
plt.axvline(r_start, color="magenta", lw=1.5)
plt.axhline(crosspoint, color="purple", lw=1.5, label=f"crosspoint @ {crosspoint}")
plt.axhline(az_start, color="darkviolet", lw=1.5, label="area of paleochannel")
plt.axhline(az_end, color="darkviolet", lw=1.5)
plt.axvspan(r_start, r_end, alpha=0.15, color="hotpink")
plt.title(f"Cross-section location in phase difference (HH-VV) ")
plt.xlabel("Range pixel")
plt.ylabel("Azimuth pixel")
plt.legend(loc="upper right")
# cbar = plt.colorbar(fraction=0.03, pad=0.02, label="Phase difference (rad)")
# cbar.set_ticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
# cbar.set_ticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
plt.ylim(amax,amin)
plt.xlim(rangemin, rangemax)
plt.tight_layout()
plt.show()



In [ ]:
range = 190
azimuth = 7980

# Limits
rangemin, rangemax = 50, 450
amin, amax = azimuth - 3000, azimuth + 3000

# Paleochannel bounds
r_start, r_end = 120, 210
az_start, az_end = 7500, 8030

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 10))

# ==========================================================
# RANGE CROSS-SECTION (fixed azimuth)
# ==========================================================

phase_range = stack_c[azimuth, :]

valid = np.isfinite(phase_range)
x = np.arange(stack_c.shape[1])[valid]
y = phase_range[valid]

ax1.plot(x, y, ".", ms=2, alpha=0.35, color="black", label="pixels")

if x.size >= 5:

    win = min(501, x.size if x.size % 2 else x.size - 1)
    win = max(5, win)

    kernel = np.ones(win) / win
    y_trend = np.convolve(y, kernel, mode="same")

    ax1.plot(x, y_trend, "r-", lw=2, label=f"trend (win={win})")

    i_min = np.nanargmin(y_trend)
    i_max = np.nanargmax(y_trend)

    ax1.scatter(
        x[i_min], y_trend[i_min],
        c="cyan", s=40,
        label=f"min = {y_trend[i_min]:.2f} rad"
    )
    ax1.scatter(
        x[i_max], y_trend[i_max],
        c="yellow", s=40,
        edgecolor="k",
        label=f"max = {y_trend[i_max]:.2f} rad"
    )

ax1.axvline(
    range,
    color="magenta",
    lw=2,
    label=f"crosspoint @ {range}"
)

ax1.axvline(
    r_start,
    color="hotpink",
    lw=2,
    label="paleochannel"
)

ax1.axvline(r_end, color="hotpink", lw=2)

ax1.axvspan(
    r_start,
    r_end,
    color="hotpink",
    alpha=0.15
)

ax1.set_xlim(rangemin, rangemax)
ax1.set_ylim(-0.1, 0.1)

ax1.set_title(
    f"HH-VV phase vs range\n(azimuth={azimuth})"
)

ax1.set_xlabel("Range index", color="purple")
ax1.set_ylabel("Phase difference (rad)")

ax1.spines["bottom"].set_color("purple")
ax1.tick_params(axis="x", colors="purple")

ax1.grid(alpha=0.3)

# ==========================================================
# AZIMUTH CROSS-SECTION (fixed range)
# ==========================================================

phase_az = stack_c[:, range]

valid = np.isfinite(phase_az)
x = np.arange(stack_c.shape[0])[valid]
y = phase_az[valid]

ax2.plot(x, y, ".", ms=2, alpha=0.35, color="black", label="pixels")

if x.size >= 5:

    win = min(501, x.size if x.size % 2 else x.size - 1)
    win = max(5, win)

    kernel = np.ones(win) / win
    y_trend = np.convolve(y, kernel, mode="same")

    ax2.plot(x, y_trend, "r-", lw=2, label=f"trend (win={win})")

    i_min = np.nanargmin(y_trend)
    i_max = np.nanargmax(y_trend)

    ax2.scatter(
        x[i_min], y_trend[i_min],
        c="cyan", s=40,
        label=f"min = {y_trend[i_min]:.2f} rad"
    )

    ax2.scatter(
        x[i_max], y_trend[i_max],
        c="yellow", s=40,
        edgecolor="k",
        label=f"max = {y_trend[i_max]:.2f} rad"
    )

ax2.axvline(
    azimuth,
    color="purple",
    lw=2,
    label=f"crosspoint @ {azimuth}"
)

ax2.axvline(
    az_start,
    color="darkviolet",
    lw=2,
    label="paleochannel"
)

ax2.axvline(
    az_end,
    color="darkviolet",
    lw=2
)

ax2.axvspan(
    az_start,
    az_end,
    color="purple",
    alpha=0.15
)

ax2.set_xlim(amin, amax)
ax2.set_ylim(-0.2, 0.2)

ax2.set_title(
    f"HH-VV phase vs azimuth\n(range={range})"
)

ax2.set_xlabel("Azimuth index", color="magenta")

ax2.spines["bottom"].set_color("magenta")
ax2.tick_params(axis="x", colors="magenta")

ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CHANNEL PHASE PROFILE  —  average phase INSIDE vs OUTSIDE the paleochannel

import numpy as np
import matplotlib.pyplot as plt

def _circ_std(a):
    a = np.asarray(a); a = a[np.isfinite(a)]
    if a.size == 0: return np.nan
    R = np.abs(np.mean(np.exp(1j*a)))
    return float(np.sqrt(-2.0*np.log(max(R, 1e-12))))

def _circ_diff(x, y):
    return float(np.angle(np.exp(1j*(x - y))))

def _circ_mean_axis(phase2d, axis, weight2d=None):
    """Weighted circular mean along an axis -> (mean_phase_1d, summed_weight_1d)."""
    m = np.isfinite(phase2d)
    z = np.where(m, np.exp(1j*phase2d), 0.0)
    w = m.astype(float) if weight2d is None else np.where(m, np.nan_to_num(weight2d, nan=0.0), 0.0)
    num = np.sum(z*w, axis=axis)
    den = np.sum(w, axis=axis)
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, np.angle(num/np.where(den > 0, den, 1)), np.nan)
    return out, den

def _wmean(vals, wts=None):
    vals = np.asarray(vals).ravel()
    m = np.isfinite(vals)
    w = m.astype(float) if wts is None else np.where(m, np.nan_to_num(np.asarray(wts).ravel(), nan=0.0), 0.0)
    den = w.sum()
    if den <= 0: return np.nan
    return float(np.angle((np.where(m, np.exp(1j*vals), 0.0)*w).sum()/den))

def channel_phase_stats(phase_map, r_start, r_end, az_start, az_end,
                        bg_pad=None, bg_gap=None, weight_map=None):
    """Average phase inside the channel box vs. two flanking background bands
    (same azimuth extent, offset in range). Returns a dict of stats + band coords."""
    H, W = phase_map.shape
    r0, r1 = sorted((int(r_start), int(r_end)))
    a0, a1 = sorted((int(az_start), int(az_end)))
    r0, r1 = max(0, r0), min(W, r1)
    a0, a1 = max(0, a0), min(H, a1)
    cw = max(1, r1 - r0)
    if bg_pad is None: bg_pad = cw                 # background band width (range)
    if bg_gap is None: bg_gap = max(5, cw // 2)    # gap channel<->background

    ch = phase_map[a0:a1, r0:r1]
    wch = None if weight_map is None else weight_map[a0:a1, r0:r1]

    lb0, lb1 = max(0, r0 - bg_gap - bg_pad), max(0, r0 - bg_gap)
    rb0, rb1 = min(W, r1 + bg_gap), min(W, r1 + bg_gap + bg_pad)
    bg = np.concatenate([phase_map[a0:a1, lb0:lb1].ravel(),
                         phase_map[a0:a1, rb0:rb1].ravel()])
    wbg = None if weight_map is None else np.concatenate(
        [weight_map[a0:a1, lb0:lb1].ravel(), weight_map[a0:a1, rb0:rb1].ravel()])

    ch_mean = _wmean(ch, wch)
    bg_mean = _wmean(bg, wbg)
    return {
        "channel_mean": ch_mean, "channel_std": _circ_std(ch), "channel_n": int(np.isfinite(ch).sum()),
        "bg_mean": bg_mean,       "bg_std": _circ_std(bg),      "bg_n": int(np.isfinite(bg).sum()),
        "contrast": _circ_diff(ch_mean, bg_mean),
        "bands": dict(chan=(r0, r1, a0, a1), left=(lb0, lb1, a0, a1), right=(rb0, rb1, a0, a1)),
    }

def channel_phase_profile(phase_map, r_start, r_end, az_start, az_end,
                          bg_pad=None, bg_gap=None, weight_map=None,
                          smooth=15, rangemin=None, rangemax=None, ylim=(-0.1, 0.1),
                          backdrop=None, label="stack_c phase diff"):
    """Cross-channel range profile (phase averaged over the channel's azimuth
    band) + inside/outside averages. Draws the profile and an ROI map."""
    s = channel_phase_stats(phase_map, r_start, r_end, az_start, az_end,
                            bg_pad=bg_pad, bg_gap=bg_gap, weight_map=weight_map)
    (r0, r1, a0, a1) = s["bands"]["chan"]
    (lb0, lb1, _, _) = s["bands"]["left"]
    (rb0, rb1, _, _) = s["bands"]["right"]

    # cross-channel profile: circular mean over the channel azimuth band, per range
    wband = None if weight_map is None else weight_map[a0:a1, :]
    prof, cnt = _circ_mean_axis(phase_map[a0:a1, :], axis=0, weight2d=wband)
    x = np.arange(phase_map.shape[1])

    if rangemin is None: rangemin = max(0, lb0 - 20)
    if rangemax is None: rangemax = min(phase_map.shape[1], rb1 + 20)

    print(f"=== Channel phase averages ({label}) ===")
    print(f"  INSIDE  channel : {s['channel_mean']:+.4f} rad  "
          f"(circ.std {s['channel_std']:.4f}, n={s['channel_n']})")
    print(f"  OUTSIDE (bg)    : {s['bg_mean']:+.4f} rad  "
          f"(circ.std {s['bg_std']:.4f}, n={s['bg_n']})")
    print(f"  CONTRAST in-out : {s['contrast']:+.4f} rad")
    print(f"  channel range [{r0},{r1}]  bg bands [{lb0},{lb1}] & [{rb0},{rb1}]  "
          f"azimuth [{a0},{a1}]")

    # --- profile plot ---
    plt.figure(figsize=(9, 5))
    m = np.isfinite(prof)
    plt.plot(x[m], prof[m], ".", ms=2, alpha=0.35, label="range profile (azimuth-averaged)")
    if smooth and smooth > 1:
        w = int(smooth) | 1
        k = np.ones(w)/w
        filled = np.where(m, prof, 0.0)
        sm = np.convolve(filled, k, mode="same")/np.maximum(np.convolve(m.astype(float), k, mode="same"), 1e-6)
        plt.plot(x, sm, "r-", lw=2, label=f"smoothed (win={w})")
    plt.axvspan(r0, r1, color="magenta", alpha=0.15, label="channel")
    plt.axvspan(lb0, lb1, color="steelblue", alpha=0.12, label="background")
    plt.axvspan(rb0, rb1, color="steelblue", alpha=0.12)
    plt.axhline(s["channel_mean"], color="magenta", ls="--", lw=1.5,
                label=f"channel mean = {s['channel_mean']:+.3f}")
    plt.axhline(s["bg_mean"], color="steelblue", ls="--", lw=1.5,
                label=f"background mean = {s['bg_mean']:+.3f}")
    plt.xlim(rangemin, rangemax); plt.ylim(*ylim)
    plt.xlabel("Range pixel"); plt.ylabel(f"{label} (rad)")
    plt.title(f"Cross-channel phase profile  (azimuth band {a0}-{a1})\n"
              f"contrast in-out = {s['contrast']:+.3f} rad")
    plt.legend(fontsize=8, loc="upper right"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # --- ROI map ---
    plt.figure(figsize=(6, 10))
    if backdrop is None:
        plt.imshow(phase_map, cmap="seismic", vmin=-np.pi, vmax=np.pi, aspect="auto")
        cbar = plt.colorbar(fraction=0.03, pad=0.02, label=f"{label} (rad)")
        cbar.set_ticks([-np.pi, 0, np.pi]); cbar.set_ticklabels([r"$-\pi$", "0", r"$\pi$"])
    else:
        plt.imshow(backdrop, cmap="gray", vmin=-30, vmax=0, aspect="auto")
        plt.colorbar(fraction=0.03, pad=0.02, label="Backscatter (dB)")
    for (bx0, bx1), col, lab in [((r0, r1), "magenta", "channel"),
                                  ((lb0, lb1), "cyan", "background"),
                                  ((rb0, rb1), "cyan", None)]:
        plt.plot([bx0, bx1, bx1, bx0, bx0], [a0, a0, a1, a1, a0], col, lw=1.8, label=lab)
    plt.ylim(amin, amax); plt.xlim(rangemin, rangemax)
    plt.xlabel("Range pixel"); plt.ylabel("Azimuth pixel")
    plt.title("Channel vs background regions"); plt.legend(loc="upper right", fontsize=8)
    plt.tight_layout(); plt.show()
    return s


In [ ]:
# --- Apply to the paleochannel ROI ---------------------------------------------
# Channel box (edit to your site). These match the site-2 numbers in this notebook;
# for the Sahara site use r_start,r_end = 1020,1040 and az_start,az_end = 15300,15600.
r_start, r_end   = 190, 210
az_start, az_end = 7900, 7600

stats = channel_phase_profile(
    stack_c, r_start, r_end, az_start, az_end,
    bg_pad=None,     # background band width in range (default = channel width)
    bg_gap=75,     # gap between channel and background (default = half channel width)
    smooth=21,       # smoothing window for the red trend line
    ylim=(-0.05, 0.05),
    backdrop=stk_db,   # set backdrop=stk_db to draw ROIs on the backscatter image instead
)

# Two-number answer:
print(f"\nIn-channel  mean phase : {stats['channel_mean']:+.4f} rad")
print(f"Background   mean phase : {stats['bg_mean']:+.4f} rad")
print(f"Difference (chan - bg)  : {stats['contrast']:+.4f} rad")
